In [36]:
from scapy.all import rdpcap, IP, UDP, TCP, PcapReader
import matplotlib.pyplot as plt
import numpy as np

In [37]:
pcap_server_fp = "/tmp/sidekick-logs/h2-eth0.pcap"
pcap_proxy_server_fp = "/tmp/sidekick-logs/p1-eth1.pcap"
pcap_client_fp = "/tmp/sidekick-logs/h1-eth0.pcap"
pcap_proxy_client_fp = "/tmp/sidekick-logs/p1-eth0.pcap"

In [38]:
server_pkts = rdpcap(pcap_server_fp)

In [39]:
client_pkts = rdpcap(pcap_client_fp)

In [40]:
proxy_clientside_pkts = rdpcap(pcap_proxy_client_fp)
proxy_serverside_pkts = rdpcap(pcap_proxy_server_fp)

In [41]:
labels = {
    "172.16.2.10" : "sender",
    "172.16.1.12" : "proxy_s",
    "172.16.1.11" : "proxy_r",
    "172.16.1.10" : "receiver"
}

# Data Transmitted

In [46]:
'''
pcaps = [server_pkts, proxy_serverside_pkts, proxy_clientside_pkts, client_pkts]
def get_first_ts(pcaps, proto):
    first_ts = None
    for pcap in pcaps:
        for pkt in pcap: 
            if (IP in pcap and proto in pcap): 
                if not first_ts:
                    first_ts = pkt.time
                else: 
                    first_ts = min(first_ts, pkt.time)
                break 
    return first_ts

first_ts = get_first_ts(pcaps, UDP)
'''

'\npcaps = [server_pkts, proxy_serverside_pkts, proxy_clientside_pkts, client_pkts]\ndef get_first_ts(pcaps, proto):\n    first_ts = None\n    for pcap in pcaps:\n        for pkt in pcap: \n            if (IP in pcap and proto in pcap): \n                if not first_ts:\n                    first_ts = pkt.time\n                else: \n                    first_ts = min(first_ts, pkt.time)\n                break \n    return first_ts\n\nfirst_ts = get_first_ts(pcaps, UDP)\n'

In [47]:
BIN_SIZE = 0.1 # 100ms

def bytes_ts(pcap, proto=UDP):
    first_ts = None
    # Flow -> { bin : nbytes }
    series = {}   # key = (src, dst), value = dict{bin_index -> bytes} 
    for pkt in pcap:
        if IP not in pkt or proto not in pkt:
            continue

        if first_ts is None:
            first_ts = pkt.time
        
        bin_idx = int((pkt.time - first_ts) // BIN_SIZE)
        size = len(bytes(pkt[proto].payload))
        key = f"{labels[pkt[IP].src]}->{labels[pkt[IP].dst]}"
        if key not in series:
            series[key] = {}
        series[key][bin_idx] = series[key].get(bin_idx, 0) + size

    # Duration of all flows (number of BIN_SIZE-second bins) 
    max_bin = 0
    for d in series.values():
        if d:
            max_bin = max(max_bin, max(d.keys()))
    # x-axis
    bins = np.arange(max_bin + 1) * BIN_SIZE

    # Fill out with zeroes if needed
    dense_series = {}
    for key, counts in series.items():
        dense_series[key] = np.zeros(max_bin + 1)
        for bin_idx, val in counts.items():
            dense_series[key][bin_idx] = val

    dense_series_mbps = {k: v * 8 / 1_000_000 / BIN_SIZE for k, v in dense_series.items()}
    return bins, dense_series_mbps

In [48]:
YMAX = 30.0

def plot_bytes_time_series(bins, dense_series, label, ax):
    
    for direction, arr in dense_series.items():
        ax.plot(bins, arr, label=str(direction))
    
    ax.set_xlabel("Time (s)")
    ax.set_ylabel(f"Mbps")
    ax.set_ylim(0, YMAX)
    ax.set_title(f"Mbps per {int(BIN_SIZE * 1000)}ms interval ({label})")
    ax.legend()

In [49]:
server_bins, server_ts = bytes_ts(server_pkts)
proxy_s_bins, proxy_s_ts = bytes_ts(proxy_serverside_pkts)

In [50]:
proxy_c_bins, proxy_c_ts = bytes_ts(proxy_clientside_pkts)
client_bins, client_ts = bytes_ts(client_pkts)

In [53]:
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes_flat = axes.flatten()

plot_bytes_time_series(server_bins, server_ts, "sender", axes_flat[0])
plot_bytes_time_series(proxy_s_bins, proxy_s_ts, "proxy-s", axes_flat[1])
plot_bytes_time_series(proxy_c_bins, proxy_c_ts, "proxy-r", axes_flat[2])
plot_bytes_time_series(client_bins, client_ts, "receiver", axes_flat[3])

plt.tight_layout()
plt.show()

# Interarrivals

In [42]:
# Source: https://blog.diogomonica.com/2011/02/01/packet-inter-arrival-time-with-scapy/
def get_ts(pcap):
    # Time flow last seen A -> B
    last_seen = {}
    # Interarrivals
    interarr = {}
    
    for p in pcap:
        if IP not in p or (UDP not in p and TCP not in p):
            continue

        src = labels[p[IP].src]
        dst = labels[p[IP].dst]
        key = f"{src}:{p[IP].sport}->{dst}:{p[IP].dport}"
        # timestamp recorded in pcap file; float - seconds since last epoch
        t = p.time
    
        if key in last_seen:
            dt = t - last_seen[key]
            interarr.setdefault(key, []).append(dt)
    
        last_seen[key] = t

    return interarr

In [43]:
def plot_ts(ts_data, label): 
    for key, dt_list in ts_data.items():
        if not dt_list: # no packets
            continue
            
        x = np.arange(len(dt_list))
        y = np.array(dt_list)

        plt.figure()
        plt.plot(x, y)
        plt.xlabel("Packet index")
        plt.ylabel("Interarrival time (s)")
        plt.title(f"Interarrival Times: {label}: {key}")
        plt.tight_layout()
        plt.show()

In [44]:
server_ts = get_ts(server_pkts)
proxy_s_ts = get_ts(proxy_serverside_pkts)
proxy_c_ts = get_ts(proxy_clientside_pkts)
client_ts = get_ts(client_pkts)

In [52]:
plot_ts(server_ts, "Server")
plot_ts(client_ts, "Proxy (Server-Side)")
plot_ts(client_ts, "Proxy (Client-Side)")
plot_ts(client_ts, "Client")